# 14 — Complete Experimental Report Generator

This notebook consolidates the adaptive-QEM experiment into a reproducible research report.

It combines:

- IBM Kingston calibration provenance
- software/environment information
- benchmark characterization
- adaptive selections
- hardware job IDs
- raw and mitigated results
- statistical comparison
- execution overhead
- publication figures/tables
- experimental limitations

**No missing experimental values are invented.**


In [ ]:
from pathlib import Path
import json, datetime
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if (ROOT / "Adaptive_QEM_IBM").exists() and not (ROOT / "data").exists():
    ROOT = ROOT / "Adaptive_QEM_IBM"

REPORT = ROOT / "results" / "reports"
REPORT.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("REPORT:", REPORT)


## 1. Load provenance and experiment artifacts

In [ ]:
def load_json(path):
    if path.exists():
        return json.loads(path.read_text(encoding="utf-8"))
    return {}

def load_csv(path):
    return pd.read_csv(path) if path.exists() else pd.DataFrame()

cal_manifest = load_json(
    ROOT / "data/calibration/ibm_kingston_calibration_manifest.json"
)

campaign_manifest = load_json(
    ROOT / "data/hardware/adaptive_qem_hardware_campaign_manifest.json"
)

selection = load_csv(
    ROOT / "data/hardware/adaptive_qem_selection_plan.csv"
)

characterization = load_csv(
    ROOT / "data/hardware/hardware_campaign_characterization.csv"
)

comparison = load_csv(
    ROOT / "data/hardware/extracted/qem_comparison_results.csv"
)

summary = load_csv(
    ROOT / "results/tables/adaptive_vs_fixed_qem_summary.csv"
)

coverage = load_csv(
    ROOT / "results/tables/adaptive_policy_coverage.csv"
)

overhead = load_csv(
    ROOT / "results/tables/qem_execution_overhead_by_circuit.csv"
)

print("Calibration manifest:", bool(cal_manifest))
print("Campaign manifest:", bool(campaign_manifest))
print("Selection rows:", len(selection))
print("Characterization rows:", len(characterization))
print("Comparison rows:", len(comparison))


## 2. Experimental status

This cell distinguishes between a **framework-complete** project and a **hardware-result-complete** project.

The manuscript should not claim experimental validation until actual IBM job results are available.


In [ ]:
status = {
    "calibration_available": bool(cal_manifest),
    "campaign_manifest_available": bool(campaign_manifest),
    "hardware_comparison_available": not comparison.empty,
    "statistical_summary_available": not summary.empty,
    "publication_figures_directory": (ROOT / "results/figures").exists(),
}

status_df = pd.DataFrame(
    [{"artifact": k, "available": v} for k, v in status.items()]
)

display(status_df)


## 3. Backend and software provenance

In [ ]:
provenance = {
    "backend": campaign_manifest.get("backend", cal_manifest.get("backend", "unknown")),
    "backend_version": cal_manifest.get("backend_version", "unknown"),
    "num_qubits": cal_manifest.get("num_qubits", "unknown"),
    "operational_at_snapshot": cal_manifest.get("operational", "unknown"),
    "qiskit": cal_manifest.get("software", {}).get("qiskit", "unknown"),
    "qiskit_ibm_runtime": cal_manifest.get("software", {}).get(
        "qiskit_ibm_runtime", "unknown"
    ),
    "shots": campaign_manifest.get("shots", "unknown"),
    "optimization_level": campaign_manifest.get("optimization_level", "unknown"),
    "seed_transpiler": campaign_manifest.get("seed_transpiler", "unknown"),
}

display(pd.DataFrame([provenance]).T.rename(columns={0:"value"}))


## 4. Calibration summary

In [ ]:
cal_file = ROOT / "data/calibration/ibm_kingston_calibration_snapshot.csv"

if cal_file.exists():
    cal = pd.read_csv(cal_file)
    calibration_summary = {}

    for col in ["t1_s", "t2_s", "readout_error",
                "prob_meas0_prep1", "prob_meas1_prep0"]:
        if col in cal.columns:
            s = pd.to_numeric(cal[col], errors="coerce").dropna()
            if len(s):
                calibration_summary[col] = {
                    "n": int(len(s)),
                    "mean": float(s.mean()),
                    "std": float(s.std()),
                    "min": float(s.min()),
                    "max": float(s.max()),
                }

    calibration_summary_df = pd.DataFrame(calibration_summary).T
    display(calibration_summary_df)
else:
    calibration_summary_df = pd.DataFrame()
    print("Calibration snapshot unavailable.")


## 5. Adaptive policy distribution

In [ ]:
if not coverage.empty:
    display(coverage)
else:
    print("Adaptive policy coverage is pending.")


## 6. Benchmark characterization

In [ ]:
if not characterization.empty:
    display(characterization)
else:
    print("Benchmark characterization is pending.")


## 7. Hardware result summary

In [ ]:
if not comparison.empty:
    display(comparison)
else:
    print("Hardware QEM comparison is pending real IBM execution.")


## 8. Statistical summary

In [ ]:
if not summary.empty:
    display(summary)
else:
    print("Statistical summary is pending.")


## 9. Execution overhead

In [ ]:
if not overhead.empty:
    display(overhead)
else:
    print("Execution-overhead dataset is pending.")


## 10. Job provenance

Every hardware result used in the manuscript should be traceable to a Runtime job ID. This section inventories the recorded job metadata files.


In [ ]:
job_dir = ROOT / "data/hardware/jobs"
job_inventory = []

if job_dir.exists():
    for f in sorted(job_dir.glob("*.json")):
        try:
            j = json.loads(f.read_text(encoding="utf-8"))
            job_inventory.append({
                "file": f.name,
                "job_id": j.get("job_id"),
                "strategy": j.get("strategy"),
                "backend": j.get("backend"),
                "shots": j.get("shots"),
                "circuits": len(j.get("circuits", [])),
            })
        except Exception as exc:
            print("Could not read", f, exc)

job_df = pd.DataFrame(job_inventory)
display(job_df)


## 11. Reproducibility manifest

The manifest captures the files that form the computational chain. It is intended to accompany the project repository and support later reruns.


In [ ]:
manifest = {
    "generated_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "backend": provenance["backend"],
    "software": {
        "qiskit": provenance["qiskit"],
        "qiskit_ibm_runtime": provenance["qiskit_ibm_runtime"],
    },
    "experiment": {
        "shots": provenance["shots"],
        "optimization_level": provenance["optimization_level"],
        "seed_transpiler": provenance["seed_transpiler"],
    },
    "artifacts": {
        "calibration": str(cal_file),
        "selection": str(ROOT / "data/hardware/adaptive_qem_selection_plan.csv"),
        "campaign": str(ROOT / "data/hardware/adaptive_qem_hardware_campaign_manifest.json"),
        "comparison": str(ROOT / "data/hardware/extracted/qem_comparison_results.csv"),
        "summary": str(ROOT / "results/tables/adaptive_vs_fixed_qem_summary.csv"),
    },
}

manifest_path = REPORT / "adaptive_qem_reproducibility_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print("Saved:", manifest_path)


## 12. Machine-readable experimental report

The JSON report preserves the key measured and provenance fields for automated manuscript generation.


In [ ]:
report = {
    "project": "Adaptive Selection of Quantum Error Mitigation Techniques for Noisy Quantum Circuits on IBM Quantum Hardware",
    "provenance": provenance,
    "status": status,
    "calibration_summary": calibration_summary_df.to_dict(orient="index")
        if not calibration_summary_df.empty else {},
    "adaptive_policy": coverage.to_dict(orient="records")
        if not coverage.empty else [],
    "benchmark_characterization": characterization.to_dict(orient="records")
        if not characterization.empty else [],
    "qem_summary": summary.to_dict(orient="records")
        if not summary.empty else [],
    "overhead": overhead.to_dict(orient="records")
        if not overhead.empty else [],
    "jobs": job_inventory,
}

report_path = REPORT / "adaptive_qem_experimental_report.json"
report_path.write_text(json.dumps(report, indent=2, default=str), encoding="utf-8")
print("Saved:", report_path)


## 13. Human-readable report

The following Markdown report can serve as the factual basis for the Results and Experimental Setup sections of the paper. It deliberately leaves unavailable hardware results as pending.


In [ ]:
lines = []

lines.append("# Adaptive QEM Experimental Report")
lines.append("")
lines.append("## Experimental identification")
lines.append("")
lines.append(
    "Study: Adaptive Selection of Quantum Error Mitigation Techniques "
    "for Noisy Quantum Circuits on IBM Quantum Hardware."
)
lines.append("")
lines.append(f"- Backend: {provenance['backend']}")
lines.append(f"- Backend version: {provenance['backend_version']}")
lines.append(f"- Backend qubits: {provenance['num_qubits']}")
lines.append(f"- Shots: {provenance['shots']}")
lines.append(f"- Optimization level: {provenance['optimization_level']}")
lines.append(f"- Transpiler seed: {provenance['seed_transpiler']}")
lines.append(f"- Qiskit: {provenance['qiskit']}")
lines.append(f"- Qiskit IBM Runtime: {provenance['qiskit_ibm_runtime']}")
lines.append("")

lines.append("## Experimental status")
lines.append("")
for k, v in status.items():
    lines.append(f"- {k}: {v}")
lines.append("")

if not coverage.empty:
    lines.append("## Adaptive policy coverage")
    lines.append("")
    lines.append(coverage.to_markdown(index=False))
    lines.append("")

if not summary.empty:
    lines.append("## QEM comparison")
    lines.append("")
    lines.append(summary.to_markdown(index=False))
    lines.append("")

if not overhead.empty:
    lines.append("## Execution overhead")
    lines.append("")
    lines.append(overhead.to_markdown(index=False))
    lines.append("")

lines.append("## Scientific limitations")
lines.append("")
lines.append(
    "The adaptive selector is a transparent rule-based baseline. "
    "Its thresholds require empirical validation on the collected hardware dataset. "
    "Results are backend-, calibration-, circuit-, and execution-condition dependent. "
    "The report does not infer universal superiority from descriptive comparisons."
)

md_report = REPORT / "adaptive_qem_experimental_report.md"
md_report.write_text("\n".join(lines), encoding="utf-8")

print("Saved:", md_report)


## 14. Final readiness gate

A manuscript-ready experimental claim requires all of the following:

- real IBM hardware job IDs;
- calibration snapshot;
- raw hardware counts;
- mitigation outputs;
- adaptive-selection records;
- statistical comparison;
- overhead accounting;
- reproducibility manifest.

If any are missing, the project is computationally prepared but the corresponding experimental claim remains pending.


In [ ]:
required = {
    "calibration_manifest": ROOT / "data/calibration/ibm_kingston_calibration_manifest.json",
    "campaign_manifest": ROOT / "data/hardware/adaptive_qem_hardware_campaign_manifest.json",
    "raw_results": ROOT / "data/hardware/extracted/ibm_sampler_v2_extracted_results.csv",
    "qem_comparison": ROOT / "data/hardware/extracted/qem_comparison_results.csv",
    "reproducibility_manifest": REPORT / "adaptive_qem_reproducibility_manifest.json",
}

gate = pd.DataFrame([
    {"artifact": name, "available": path.exists()}
    for name, path in required.items()
])

display(gate)
print("Experimental report gate:", "READY" if gate["available"].all() else "PENDING HARDWARE DATA")
